# Chunking strategies: podcast vs PDF
**Lab:** *Slice docs the way Google wishes you would*

Run top to bottom. Edit the **CONFIG** cell first. The last cell writes `lab_proof.md` from your own results.

## Step 1: Setup and data loading

In [1]:
%pip install -q langchain-text-splitters langchain-community langchain-core pypdf tiktoken openai python-dotenv scikit-learn matplotlib pandas
# Step 5 (optional): %pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, re, json, bisect
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
import tiktoken

load_dotenv()  # reads OPENAI_API_KEY from .env (never hard-code keys)

# ---------------- CONFIG: edit these ----------------
AUDIO_PATH = "The_Blueprint_For_Trustworthy_AI.m4a"       # your podcast audio file
PDF_SOURCE = "ai_hleg_ethics_guidelines_for_trustworthy_ai-en_87F84A41-A6E8-F38C-BFF661481B40077B_60419 (1).pdf"      # local path or https:// URL to the PDF
QUERY = "What are three components of trustworthy AI"
TRANSCRIBE_WITH = "openai"       # "openai" (whisper-1 API, 25 MB file limit) or "local" (pip install openai-whisper)
RUN_SEMANTIC = False             # Step 5 (optional): needs sentence-transformers
# -----------------------------------------------------
TRANSCRIPT_PATH = Path("transcript.txt")
SEGMENTS_PATH = Path("transcript_segments.json")

enc = tiktoken.get_encoding("cl100k_base")   # GPT-4 / GPT-3.5 tokenizer (GPT-4o uses o200k_base)
def n_tokens(text): return len(enc.encode(text))
print("Imports OK")

Imports OK


In [21]:
AUDIO_PATH = Path("audio_small.m4a")

def transcribe(audio_path):
    """Transcribe once, save text + timestamped segments locally, reuse on re-runs."""
    if SEGMENTS_PATH.exists() or TRANSCRIPT_PATH.exists():
        print("Using saved transcript (delete transcript files to re-transcribe).")
        return
    if TRANSCRIBE_WITH == "local":
        import whisper
        result = whisper.load_model("base").transcribe(str(audio_path))
        segments = [{"start": s["start"], "end": s["end"], "text": s["text"]} for s in result["segments"]]
    else:
        from openai import OpenAI
        client = OpenAI(max_retries=5, timeout=600)
        with open(audio_path, "rb") as f:
            result = client.audio.transcriptions.create(
                model="whisper-1", file=f, response_format="verbose_json")
        segments = [{"start": s.start, "end": s.end, "text": s.text} for s in result.segments]
    SEGMENTS_PATH.write_text(json.dumps(segments, indent=1), encoding="utf-8")
    TRANSCRIPT_PATH.write_text(" ".join(s["text"].strip() for s in segments), encoding="utf-8")
    print(f"Saved {len(segments)} segments -> {TRANSCRIPT_PATH}")

transcribe(AUDIO_PATH)

Saved 445 segments -> transcript.txt


In [ ]:
import json
from pathlib import Path
from openai import OpenAI

AUDIO_PATH = Path("audio_small.m4a")
print("audio:", AUDIO_PATH.resolve(), "exists:", AUDIO_PATH.exists(),
      f"{AUDIO_PATH.stat().st_size/1e6:.1f} MB" if AUDIO_PATH.exists() else "")

if not TRANSCRIPT_PATH.exists():
    client = OpenAI(max_retries=5, timeout=600)
    with open(AUDIO_PATH, "rb") as f:
        result = client.audio.transcriptions.create(
            model="whisper-1", file=f, response_format="verbose_json")
    segments = [{"start": s.start, "end": s.end, "text": s.text} for s in result.segments]
    SEGMENTS_PATH.write_text(json.dumps(segments, indent=1), encoding="utf-8")
    TRANSCRIPT_PATH.write_text(" ".join(s["text"].strip() for s in segments), encoding="utf-8")
    print(f"Saved {len(segments)} segments")

print("transcript:", TRANSCRIPT_PATH.resolve(), "exists:", TRANSCRIPT_PATH.exists())

Audio size: 3.8 MB (OpenAI limit: 25 MB)
Saved 445 segments -> transcript.txt


In [16]:
# Podcast -> ONE Document (Whisper gives one long paragraph). Keep segment offsets so every chunk gets a timestamp.
if SEGMENTS_PATH.exists():
    segments = json.loads(SEGMENTS_PATH.read_text(encoding="utf-8"))
    parts, seg_offsets, seg_starts, pos = [], [], [], 0
    for s in segments:
        t = s["text"].strip()
        seg_offsets.append(pos); seg_starts.append(s["start"])
        parts.append(t); pos += len(t) + 1
    transcript = " ".join(parts)
else:
    segments, seg_offsets, seg_starts = [], [], []
    transcript = TRANSCRIPT_PATH.read_text(encoding="utf-8")

def timestamp_at(char_idx):
    if not seg_offsets or char_idx is None or char_idx < 0:
        return None
    i = max(bisect.bisect_right(seg_offsets, char_idx) - 1, 0)
    m, s = divmod(int(seg_starts[i]), 60)
    return f"{m:02d}:{s:02d}"

podcast_docs = [Document(page_content=transcript, metadata={"source": str(AUDIO_PATH), "doc_type": "podcast"})]

# PDF -> one Document per page (page number kept in metadata)
pdf_docs = [d for d in PyPDFLoader(PDF_SOURCE).load() if d.page_content.strip()]
for d in pdf_docs:
    d.metadata["doc_type"] = "pdf"

DOCS = {"podcast": podcast_docs, "pdf": pdf_docs}
pdf_text = "\n".join(d.page_content for d in pdf_docs)
SOURCE_INFO = {
    "podcast": {"chars": len(transcript), "tokens": n_tokens(transcript),
                "blank_lines": transcript.count("\n\n"), "newlines": transcript.count("\n")},
    "pdf": {"pages": len(pdf_docs), "chars": len(pdf_text), "tokens": n_tokens(pdf_text),
            "blank_lines": sum(d.page_content.count("\n\n") for d in pdf_docs),
            "newlines": sum(d.page_content.count("\n") for d in pdf_docs)},
}
for k, v in SOURCE_INFO.items():
    print(k, v)
print("\nTranscript start:", transcript[:300])
print("\nPDF page 1 start:", pdf_docs[0].page_content[:300])

FileNotFoundError: [Errno 2] No such file or directory: 'transcript.txt'

## Steps 2-5: Chunking strategies
- **Fixed** uses `separator=""`, so it cuts every N characters no matter what. (With the default separator `"\n\n"` it is *not* fixed-size; see the gotcha cell below.)
- **Recursive** tries `\n\n` → `\n` → space → character.
- **Recursive-1000-sentences** uses a custom separator priority (sentence ends before spaces). `keep_separator="end"` keeps the `.` on the left chunk; with the default (`"start"`) every chunk would *begin* with `". "`.
- **Token** counts `cl100k_base` tokens, not characters.
- All splitters use `add_start_index=True`, so every chunk keeps its offset. That offset gives each podcast chunk a timestamp.

In [ ]:
SENTENCE_SEPS = ["\n\n", "\n", ". ", "? ", "! ", " ", ""]

def split_with(splitter):
    return lambda docs: splitter.split_documents(docs)

STRATEGIES = {
    # Step 2: fixed size (sizes 500/1000/2000, overlaps 0/50/100)
    "Fixed-500":  split_with(CharacterTextSplitter(separator="", chunk_size=500,  chunk_overlap=0,   add_start_index=True)),
    "Fixed-1000": split_with(CharacterTextSplitter(separator="", chunk_size=1000, chunk_overlap=50,  add_start_index=True)),
    "Fixed-2000": split_with(CharacterTextSplitter(separator="", chunk_size=2000, chunk_overlap=100, add_start_index=True)),
    # Step 3: recursive (default separators), then custom separator priority
    "Recursive-500":  split_with(RecursiveCharacterTextSplitter(chunk_size=500,  chunk_overlap=50,  add_start_index=True)),
    "Recursive-1000": split_with(RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, add_start_index=True)),
    "Recursive-2000": split_with(RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100, add_start_index=True)),
    "Recursive-1000-sentences": split_with(RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=100, separators=SENTENCE_SEPS, keep_separator="end", add_start_index=True)),
    # Step 4: token based
    "Token-500":  split_with(TokenTextSplitter(encoding_name="cl100k_base", chunk_size=500,  chunk_overlap=50,  add_start_index=True)),
    "Token-1000": split_with(TokenTextSplitter(encoding_name="cl100k_base", chunk_size=1000, chunk_overlap=100, add_start_index=True)),
    "Recursive-Token-500": split_with(RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base", chunk_size=500, chunk_overlap=50,
        separators=SENTENCE_SEPS, keep_separator="end", add_start_index=True)),
}
print(len(STRATEGIES), "strategies")

In [ ]:
# Step 5 (optional): semantic chunking. Split where neighbouring sentences stop being similar, with a size cap.
def semantic_chunks(docs, max_chars=1500, pct=10):
    from sentence_transformers import SentenceTransformer
    import numpy as np
    model = SentenceTransformer("all-MiniLM-L6-v2")
    out = []
    for d in docs:
        text = d.page_content
        sents = [s for s in re.split(r"(?<=[.?!])\s+", text) if s.strip()]
        if len(sents) < 2:
            out.append(Document(page_content=text.strip(), metadata={**d.metadata, "start_index": 0}))
            continue
        emb = model.encode(sents, normalize_embeddings=True)
        sims = (emb[:-1] * emb[1:]).sum(axis=1)          # cosine similarity of sentence i and i+1
        cut = np.percentile(sims, pct)                     # lowest `pct`% similarities = topic shifts
        chunk, start, cursor = "", 0, 0
        def flush():
            if chunk:
                out.append(Document(page_content=chunk, metadata={**d.metadata, "start_index": start}))
        for i, s in enumerate(sents):
            idx = text.find(s, cursor); idx = cursor if idx < 0 else idx
            if chunk and len(chunk) + len(s) + 1 > max_chars:
                flush(); chunk = ""
            if not chunk:
                start = idx
            chunk = f"{chunk} {s}" if chunk else s
            cursor = idx + len(s)
            if i < len(sims) and sims[i] <= cut:
                flush(); chunk = ""
        flush()
    return out

if RUN_SEMANTIC:
    STRATEGIES["Semantic"] = semantic_chunks

In [ ]:
# Run every strategy on both sources and tag every chunk with an id (+ timestamp for the podcast)
CHUNKS = {}
for doc_type, docs in DOCS.items():
    for name, fn in STRATEGIES.items():
        chunks = [c for c in fn(docs) if c.page_content.strip()]
        for i, c in enumerate(chunks):
            c.metadata["chunk_id"] = f"{doc_type}|{name}|{i}"
            if doc_type == "podcast":
                c.metadata["timestamp"] = timestamp_at(c.metadata.get("start_index"))
        CHUNKS[(doc_type, name)] = chunks
print({f"{k[0]}/{k[1]}": len(v) for k, v in CHUNKS.items()})

In [ ]:
# Gotcha (Step 2): CharacterTextSplitter's DEFAULT separator is "\n\n". It splits on blank lines, then merges.
# A Whisper transcript has no blank lines, so the podcast stays one huge chunk (LangChain logs a warning).
default_split = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_documents(podcast_docs)
GOTCHA = {"n_chunks": len(default_split), "max_chars": max(len(c.page_content) for c in default_split)}
print(f"Default CharacterTextSplitter on the podcast: {GOTCHA['n_chunks']} chunk(s), "
      f"largest = {GOTCHA['max_chars']:,} chars (target 1,000)")

## Steps 6-7: Statistics, distributions, boundary quality

In [ ]:
def pct_clean_boundaries(chunks):
    """% of chunks ending with . ? ! (ignores closing quotes/brackets; safe on empty chunks)."""
    ends = [c.page_content.strip().rstrip("\"')]”’") for c in chunks]
    ends = [e for e in ends if e]
    return 100 * sum(e[-1] in ".?!" for e in ends) / len(ends) if ends else 0.0

rows = []
for (doc_type, name), chunks in CHUNKS.items():
    if not chunks:                      
        continue
    sizes = [len(c.page_content) for c in chunks]
    rows.append({"doc": doc_type, "strategy": name, "# chunks": len(chunks),
                 "avg chars": round(sum(sizes) / len(sizes)), "min": min(sizes), "max": max(sizes),
                 "avg tokens": round(sum(n_tokens(c.page_content) for c in chunks) / len(chunks)),
                 "% clean endings": round(pct_clean_boundaries(chunks), 1)})
stats = pd.DataFrame(rows)
def stat(doc, name, col):
    return stats[(stats["doc"] == doc) & (stats["strategy"] == name)][col].iloc[0]
stats

In [ ]:
show = ["Fixed-500", "Recursive-1000", "Token-500"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for r, doc_type in enumerate(DOCS):
    for c, name in enumerate(show):
        sizes = [len(ch.page_content) for ch in CHUNKS[(doc_type, name)]]
        axes[r, c].hist(sizes, bins=20)
        axes[r, c].set_title(f"{doc_type}: {name}")
        axes[r, c].set_xlabel("Chunk size (chars)")
        axes[r, c].set_ylabel("Frequency")
plt.tight_layout(); plt.savefig("chunk_size_distributions.png", dpi=120); plt.show()

ax = stats.pivot(index="strategy", columns="doc", values="% clean endings").plot.barh(figsize=(8, 6))
ax.set_xlabel("% of chunks ending on . ? !"); ax.set_title("Clean sentence endings by strategy")
plt.tight_layout(); plt.savefig("clean_endings.png", dpi=120); plt.show()

In [ ]:
# Where do chunks break? First/last 50 chars of a few chunks
def show_boundaries(doc_type, name, n=3):
    print(f"--- {doc_type} / {name} ---")
    for ch in CHUNKS[(doc_type, name)][:n]:
        t = ch.page_content.strip()
        print(f"  START: {t[:50]!r}\n  END:   {t[-50:]!r}\n")

for doc_type in DOCS:
    for name in ["Fixed-500", "Recursive-1000", "Recursive-1000-sentences"]:
        show_boundaries(doc_type, name)

## Grounding trace: one query → retrieved chunks → answer → check
TF-IDF retrieval (no API key needed) over the podcast and PDF chunks together. The same query runs on fixed chunks and on sentence-aware chunks so you can compare the evidence. The answer may only use the retrieved chunks and must cite them as `[C1]`, `[C2]` and so on.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RETRIEVAL_STRATEGIES = ["Fixed-500", "Recursive-1000-sentences"]
ANSWER_STRATEGY = "Recursive-1000-sentences"

def where(meta):
    if meta.get("doc_type") == "pdf":
        return f"PDF p.{meta.get('page', 0) + 1}"
    return f"podcast @{meta.get('timestamp') or 'n/a'}"

def retrieve(query, name, k=3):
    chunks = CHUNKS[("podcast", name)] + CHUNKS[("pdf", name)]
    texts = [c.page_content for c in chunks]
    vec = TfidfVectorizer(stop_words="english").fit(texts)
    scores = cosine_similarity(vec.transform([query]), vec.transform(texts))[0]
    return [(chunks[i], float(scores[i])) for i in scores.argsort()[::-1][:k]]

def answer(query, hits):
    context = "\n\n".join(f"[C{i+1}] ({where(c.metadata)}) {c.page_content}" for i, (c, _) in enumerate(hits))
    if not os.getenv("OPENAI_API_KEY"):
        return "extractive (no API key)", hits[0][0].page_content.strip()[:500] + " [C1]"
    from openai import OpenAI
    r = OpenAI().chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
        {"role": "system", "content": "Answer ONLY from the context. Put the chunk id, like [C1], after every claim. "
                                      "If the context does not contain the answer, reply exactly: Not in the sources."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}])
    return "gpt-4o-mini", r.choices[0].message.content

def check(ans, hits):
    cited = sorted({int(n) for n in re.findall(r"\[C(\d+)\]", ans)})
    return {"cited": cited,
            "citations_valid": bool(cited) and all(1 <= n <= len(hits) for n in cited),
            "refused": "Not in the sources" in ans,
            "top_score": round(hits[0][1], 3)}

TRACE = {name: retrieve(QUERY, name) for name in RETRIEVAL_STRATEGIES}
for name, hits in TRACE.items():
    print(f"\n=== {name} ===")
    for i, (c, s) in enumerate(hits, 1):
        print(f"[C{i}] {where(c.metadata)} | score {s:.3f} | {c.metadata['chunk_id']}\n     {c.page_content[:250]!r}")

ANSWER_MODEL, FINAL_ANSWER = answer(QUERY, TRACE[ANSWER_STRATEGY])
CHECK = check(FINAL_ANSWER, TRACE[ANSWER_STRATEGY])
print(f"\nFINAL ANSWER ({ANSWER_MODEL}, {ANSWER_STRATEGY}):\n{FINAL_ANSWER}\n\nCHECK: {CHECK}")
if CHECK["top_score"] == 0:
    print("WARNING: top score is 0, so the query shares no words with any chunk. Pick a query your sources answer.")

## Step 8: Recommendations + write `lab_proof.md`

In [ ]:
def md_table(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    out += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(out)

def clean(text, n=220):
    return re.sub(r"\s+", " ", text).strip()[:n].replace("|", "\\|")

def best_for(doc_type):
    cand = stats[(stats["doc"] == doc_type) & ~stats["strategy"].str.startswith("Fixed")]
    return cand.sort_values(["% clean endings", "avg chars"], ascending=[False, True]).iloc[0]

P, D = "podcast", "pdf"
bp, bd = best_for(P), best_for(D)
fixed_hits = TRACE["Fixed-500"]
_ft = fixed_hits[0][0].page_content.strip()
_cut_end = _ft.rstrip("\"')]\u201d\u2019")[-1:] not in ".?!"
_cut_start = not _ft[:1].isupper()
if _cut_end:
    fixed_cut = f'The top Fixed-500 hit for the query ends mid-sentence: "…{clean(_ft[-80:], 80)}"'
elif _cut_start:
    fixed_cut = f'The top Fixed-500 hit for the query starts mid-sentence: "{clean(_ft[:80], 80)}…"'
else:
    fixed_cut = (f"The top Fixed-500 hit happens to be clean, but {100 - stat(P,'Fixed-500','% clean endings'):.1f}% of podcast "
                 f"and {100 - stat(D,'Fixed-500','% clean endings'):.1f}% of PDF Fixed-500 chunks end mid-sentence.")
better_fixed = P if stat(P, "Fixed-500", "% clean endings") >= stat(D, "Fixed-500", "% clean endings") else D

L = []
L += ["# Lab proof: chunking strategies for a podcast and a PDF", "",
      "**Notebook:** `chunking_strategies.ipynb` (run top to bottom; its last cell generates this file).", "",
      "## 1. Sources", "",
      "| Source | Loaded as | Size | Structure |", "|---|---|---|---|",
      f"| Podcast | `{AUDIO_PATH}` → Whisper transcript (`transcript.txt`, {len(segments)} timestamped segments) | "
      f"{SOURCE_INFO[P]['chars']:,} chars / {SOURCE_INFO[P]['tokens']:,} tokens | {SOURCE_INFO[P]['blank_lines']} blank lines, {SOURCE_INFO[P]['newlines']} line breaks |",
      f"| PDF | `{PDF_SOURCE}` via PyPDFLoader, one Document per page | {SOURCE_INFO[D]['pages']} pages, "
      f"{SOURCE_INFO[D]['chars']:,} chars / {SOURCE_INFO[D]['tokens']:,} tokens | {SOURCE_INFO[D]['blank_lines']} blank lines, {SOURCE_INFO[D]['newlines']} line breaks |", "",
      "## 2. Workflow", "",
      "- **INPUT:** podcast audio (transcribed once with Whisper, saved locally) and a PDF (one Document per page).",
      "- **DECIDE:** retrieval path. Chunk with each strategy, index the chunks with TF-IDF, retrieve the top 3 for one query.",
      "- **BUILD:** split → statistics → plots → retrieve → answer only from the retrieved chunks, with `[C#]` citations.",
      "- **VERIFY:** every chunk carries `source`, `chunk_id`, `start_index` and a PDF page number or a podcast timestamp; citations are checked against the retrieved set.",
      "- **EXPLAIN:** failure and limitation in section 6.", "",
      "```python",
      'CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=0)            # Fixed-500',
      "RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)            # Recursive-1000",
      'RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100,             # Recursive-1000-sentences',
      '    separators=["\\n\\n", "\\n", ". ", "? ", "! ", " ", ""], keep_separator="end")',
      'TokenTextSplitter(encoding_name="cl100k_base", chunk_size=500, chunk_overlap=50) # Token-500',
      "```", "",
      "## 3. Chunk statistics", "", md_table(stats), "",
      "![Chunk size distributions](chunk_size_distributions.png)", "", "![Clean endings](clean_endings.png)", "",
      "## 4. Analysis (from the numbers above)", "",
      f"- **Does fixed-size break sentences?** Yes. Fixed-500 ends on `. ? !` in only {stat(P,'Fixed-500','% clean endings')}% of podcast chunks "
      f"and {stat(D,'Fixed-500','% clean endings')}% of PDF chunks; it cuts mid-word. The **{better_fixed}** handles fixed-size chunking better.",
      f"- **Recursive vs fixed (1000):** clean endings {stat(P,'Recursive-1000','% clean endings')}% vs {stat(P,'Fixed-1000','% clean endings')}% (podcast), "
      f"{stat(D,'Recursive-1000','% clean endings')}% vs {stat(D,'Fixed-1000','% clean endings')}% (PDF).",
      f"- **Structure:** the transcript has {SOURCE_INFO[P]['blank_lines']} blank lines, so default recursive splitting falls back to spaces. "
      f"Putting sentence ends first in the separator list raises podcast clean endings to {stat(P,'Recursive-1000-sentences','% clean endings')}%. "
      f"The PDF has {SOURCE_INFO[D]['blank_lines']} blank lines and {SOURCE_INFO[D]['newlines']} line breaks for recursive splitting to use.",
      f"- **Token vs character:** Token-500 chunks average {stat(P,'Token-500','avg chars')} chars (podcast) and {stat(D,'Token-500','avg chars')} chars (PDF), "
      f"so the same token budget holds a different amount of text per source. Token limits map directly to the LLM context window; character limits do not.", "",
      "## 5. Grounding trace (one query)", "", f"**Query:** {QUERY}", ""]
for name, hits in TRACE.items():
    L += [f"**Retrieved with {name}:**", "", "| # | chunk_id | where | TF-IDF score | evidence (start of chunk) |", "|---|---|---|---|---|"]
    L += [f"| C{i} | `{c.metadata['chunk_id']}` | {where(c.metadata)} | {s:.3f} | {clean(c.page_content)} |" for i, (c, s) in enumerate(hits, 1)]
    L += [""]
L += [f"**Final output** ({ANSWER_MODEL}, context = {ANSWER_STRATEGY} top 3):", ""]
L += ["> " + line for line in FINAL_ANSWER.strip().splitlines()] + [""]
L += ["**Verification:**", "",
      f"- Citations in the answer: {CHECK['cited']}; all point to retrieved chunks: **{CHECK['citations_valid']}**; model refused: {CHECK['refused']}.",
      f"- Trace for the reviewer: [C1] = `{TRACE[ANSWER_STRATEGY][0][0].metadata['chunk_id']}` at **{where(TRACE[ANSWER_STRATEGY][0][0].metadata)}** (top score {CHECK['top_score']}).", "",
      "## 6. Failure and limitation", "",
      f"1. **Default `CharacterTextSplitter` is not fixed-size.** Its default separator is `\"\\n\\n\"`. The transcript has "
      f"{SOURCE_INFO[P]['blank_lines']} blank lines, so it returned {GOTCHA['n_chunks']} chunk(s), the largest {GOTCHA['max_chars']:,} chars against a 1,000 target. "
      "Fix: `separator=\"\"` for true fixed-size, or recursive splitting with sentence separators.",
      f"2. **Fixed chunks cut evidence mid-sentence.** {fixed_cut}",
      "3. **Where a plausible but unsupported answer can appear:** TF-IDF matches words, not meaning. If the top chunk is a half-sentence, "
      "or the answer spans two chunks or two PDF pages (chunks never cross a page here), the LLM can fill the gap from its own knowledge. "
      "Guards used: answer only from context, a `[C#]` citation on every claim, \"Not in the sources\" as the fallback, and the citation check above. "
      "Not covered: the check confirms that a cited chunk exists, not that it supports the claim.", "",
      "## 7. Recommendations", "",
      "### For PDF documents:",
      f"**Recommended strategy:** {bd['strategy']}", "**Reasoning:**",
      f"- Highest clean-ending rate for the PDF among non-fixed strategies ({bd['% clean endings']}%), versus {stat(D,'Fixed-500','% clean endings')}% for Fixed-500.",
      "- Uses the PDF's own line and paragraph breaks; page numbers stay in metadata, so every answer can cite a page.",
      f"- Size: about {bd['avg chars']} chars / {bd['avg tokens']} tokens per chunk, with ~10% overlap so sentences cut at a boundary appear in both chunks.", "",
      "### For podcast transcripts:",
      f"**Recommended strategy:** {bp['strategy']}", "**Reasoning:**",
      f"- The transcript has no paragraph breaks, so sentence-level separators carry the structure: {bp['% clean endings']}% clean endings "
      f"versus {stat(P,'Fixed-500','% clean endings')}% for Fixed-500.",
      "- Each chunk maps back to a timestamp, so the reviewer can jump to that point in the audio.",
      f"- Size: about {bp['avg chars']} chars / {bp['avg tokens']} tokens per chunk, with overlap so a speaker's point is not split from its context.", "",
      "### Trade-offs summary:",
      "| Strategy | Pros | Cons | Best for |", "|---|---|---|---|",
      "| Fixed-size | Simple, predictable | Breaks context, cuts mid-word | Uniform content |",
      "| Recursive | Preserves structure | Needs the right separators (transcripts have none) | Structured docs |",
      "| Token-based | Accurate for LLM limits | Requires a matching tokenizer | LLM integration |",
      "| Semantic | Meaning-based | Computationally expensive, needs an embedding model | Complex content |", ""]

Path("lab_proof.md").write_text("\n".join(L), encoding="utf-8")
print("Wrote lab_proof.md. Read it and adjust the wording before you submit.")

In [19]:
import os
from pathlib import Path
print("working dir:", os.getcwd())
for p in [AUDIO_PATH, SEGMENTS_PATH, TRANSCRIPT_PATH]:
    print(Path(p).resolve(), "exists:", Path(p).exists())

working dir: /Users/muratcomez/labs/lab-chunking-strategies
/Users/muratcomez/labs/lab-chunking-strategies/The_Blueprint_For_Trustworthy_AI.m4a exists: True
/Users/muratcomez/labs/lab-chunking-strategies/transcript_segments.json exists: False
/Users/muratcomez/labs/lab-chunking-strategies/transcript.txt exists: False
